# MSTL Decomposition of Electricity Load

Apply MSTL (Multiple Seasonal-Trend decomposition using LOESS) to the hourly electricity load series, extracting trend, 24-hour daily seasonality, 168-hour weekly seasonality, and residual components.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import MSTL

In [ ]:
def mstl_decomposition(csv_file, target_column='load'):
    """MSTL decomposition for time series data."""
    df = pd.read_csv(csv_file)
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
    df = df.asfreq('h')

    print(f'Data shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    print(f'Date range: {df.index.min()} to {df.index.max()}')

    series = df[target_column].dropna()
    mstl = MSTL(series, periods=[24, 168])
    result = mstl.fit()

    fig, axes = plt.subplots(5, 1, figsize=(14, 12))

    axes[0].plot(series.index, series.values)
    axes[0].set_title(f'Original {target_column.title()} Series')
    axes[0].set_ylabel(target_column.title())
    axes[0].set_xticks([])

    axes[1].plot(result.trend.index, result.trend.values, color='#3B75AF')
    axes[1].set_title('Trend Component')
    axes[1].set_ylabel('Trend')
    axes[1].set_xticks([])

    axes[2].plot(result.seasonal.iloc[:, 0].index, result.seasonal.iloc[:, 0].values, color='#EA3728')
    axes[2].set_title('Daily Seasonal Component (24-hour cycle)')
    axes[2].set_ylabel('Daily Seasonal')
    axes[2].set_xticks([])

    axes[3].plot(result.seasonal.iloc[:, 1].index, result.seasonal.iloc[:, 1].values, color='#EA3728')
    axes[3].set_title('Weekly Seasonal Component (168-hour cycle)')
    axes[3].set_ylabel('Weekly Seasonal')
    axes[3].set_xticks([])

    axes[4].plot(result.resid.index, result.resid.values, color='#EF8636')
    axes[4].set_title('Residuals')
    axes[4].set_ylabel('Residuals')
    axes[4].set_xlabel('Time')

    plt.savefig('load_mstl_decomposition.png', transparent=True, bbox_inches='tight')
    plt.tight_layout()
    plt.show()

    return result, df

In [ ]:
load_result, df = mstl_decomposition('elec-temp.csv', target_column='load')